In [39]:
import pandas as pd
import numpy as np
import math
import random
from itertools import combinations
import scipy
from scipy.optimize import curve_fit, least_squares
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm



In [2]:
np.random.seed(42)

In [91]:
n=1600
d=40
w = np.random.uniform(low=0, high=1, size=(3, d))
a = np.random.normal(loc=3, scale=1, size=(d))

def get_path(verts):
    row = 0
    col = 0
    outs = []
    for i in range(8):
        if i in verts:
            outs.append(9*row + col + 4)
            row += 1
        else:
            outs.append(9*row + col)
            col += 1
    z = np.zeros(40, dtype=int)
    z[outs] = 1
    return z

feasiblepaths = np.zeros(shape=(70, 40))
i=0
for comb in combinations(range(8), 4):
    feasiblepaths[i] = get_path(comb)
    i += 1

def f(x, theta):
    return theta[0] + x[0]*theta[1] + x[1]*theta[2] + x[2]*theta[3]

def getdata(n, paths, w, a, noiserange = 0.5, impute=False):
    theta = np.vstack([a, w])
    noise = np.random.uniform(low=-noiserange, high=noiserange, size = (n, 1))
    X = np.random.normal(size = (n, 3))
    Y = np.empty((n, d))
    for i in range(n):
        Y[i] = f(X[i], theta) + noise[i]

    C = np.zeros(shape=(n, 1))
    Z = np.zeros(shape=(n, 40))
    for i in range(n):
        if impute == True:
            path = paths[i % len(paths)]
        else:
            path = paths[np.random.choice(range(len(paths)))]
        Z[i] = path
        C[i, 0] = Y[i] @ path
    return X, Y, Z, C

def solmap(y):
    min = np.inf
    out = None
    for z in feasiblepaths:
        if np.dot(y, z) <= min:
            min = np.dot(y, z)
            out = z
    return out

def pgcsubgradient(fx, y, h):
    return (solmap(fx + h * y) - solmap(fx - h * y)) / (2*h)

def derivtheta(f):
    out = np.ones(shape=(4, 40))
    out[1] = f[0]
    out[2] = f[1]
    out[3] = f[2]
    return out

def nuisancef(x, theta_opt):
    return f(x, theta_opt)

def thetadm(X, Z, C, f, Sigma):
    return f(X)

def thetadr(X, Z, C, f, Sigma):
    return np.linalg.inv(Sigma + np.identity(40)) @ Z * C

def thetadr(X, Z, C, f, theta, Sigma, reg=True):
    if reg == True:
        return f(X, theta) + np.linalg.inv(Sigma + np.identity(40)) @ Z * (C - np.dot(Z, f(X, theta)))
    else:
        return f(X, theta) + np.linalg.inv(Sigma) @ Z * (C - np.dot(Z, f(X, theta)))
    
def residual(w, c, z, x):
    w = w.reshape(4, 40)
    preds = []
    for i in range(len(x)):
        f_val = f(x[i], w)
        preds.append(np.dot(z[i], f_val))
    preds = np.array(preds).reshape(len(x), 1)
    return (c - preds).flatten()

def bandit_SGD(X, Z, C, h = 1, lr=0.01, epochs=100, reg=True):

    theta0 = np.random.randn(4 * 40)  # initial guess

    result = least_squares(residual, theta0, args=(C, Z, X))

    theta_opt = result.x.reshape(4, 40)

    Gram = np.zeros(shape=(40, 40))
    for i in range(len(Z)):
        
        Gram += Z[i].reshape(40, 1) @ Z[i].reshape(1, 40)
    Gram /= len(Z)
    theta = np.random.randn(4, 40) # Initialize weights
    for epoch in tqdm(range(epochs)):
        # Shuffle data for true stochastic behavior
        indices = np.random.permutation(len(Z))
        for i in indices:
            xi, zi, ci = X[i], Z[i], C[i]

            # 2. Use the custom gradient to update weights
            grad = (derivtheta(f(xi, theta)) * pgcsubgradient(f(xi, theta), thetadr(xi, zi, ci, nuisancef, theta_opt, Gram, reg), h))

            theta += lr * grad
    return theta

In [52]:
exploredpaths = np.empty((0, 40))
unexploredpaths = np.empty((0, 40))
unexplorable = [7, 11, 12, 16]

for comb in combinations(range(8), 4):
    path = get_path(comb)
    if np.any([path[i] == 1 for i in unexplorable]):
        unexploredpaths = np.vstack([unexploredpaths, path])
    else:
        exploredpaths = np.vstack([exploredpaths, path])


In [53]:
X, Y, Z, C = getdata(n, exploredpaths)
X_imp, Y_imp, Z_imp, C_imp = getdata(n//2, unexploredpaths, noiserange = 3)
X_train = np.vstack([X, X_imp])
Y_train = np.vstack([Y, Y_imp])
Z_train = np.vstack([Z, Z_imp])
C_train = np.vstack([C, C_imp])

X_test, Y_test, Z_test, C_test = getdata(n, feasiblepaths)

In [54]:
theta = bandit_SGD(X_train, Z_train, C_train)
cost = np.sum([np.dot(Y_test[i], solmap(f(X_test[i], theta))) for i in range(n)]) / n
optcost = np.sum([np.dot(Y_test[i], solmap(Y_test[i])) for i in range(n)]) / n
print((cost - optcost)/optcost)

  0%|          | 0/100 [00:00<?, ?it/s]

0.19131670562519418


In [94]:
out = {}
noises = [0, 0.5, 1, 1.5, 2]
n = 1600
np.random.seed(42)
w = np.random.uniform(low=0, high=1, size=(3, d))
a = np.random.normal(loc=3, scale=1, size=(d))
X, Y, Z, C = getdata(n, exploredpaths, w, a)
X_test, Y_test, Z_test, C_test = getdata(n, feasiblepaths, w, a)
for noise in noises:
    tot = 0
    for trial in range(10):
        X_imp, Y_imp, Z_imp, C_imp = getdata(100, unexploredpaths, w, a, noiserange=noise, impute=True)
        X_train = np.vstack([X, X_imp])
        Y_train = np.vstack([Y, Y_imp])
        Z_train = np.vstack([Z, Z_imp])
        C_train = np.vstack([C, C_imp])
        theta = bandit_SGD(X_train, Z_train, C_train, reg=False)
        cost = np.sum([np.dot(Y_test[i], solmap(f(X_test[i], theta))) for i in range(n)]) / n
        optcost = np.sum([np.dot(Y_test[i], solmap(Y_test[i])) for i in range(n)]) / n
        tot += (cost - optcost)/optcost
    out[noise] = tot/10


  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

...

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

In [75]:
X_imp, Y_imp, Z_imp, C_imp = getdata(100, unexploredpaths, 0 * w, 0 * a, impute=True)

In [82]:
n = 1600
w = np.random.uniform(low=0, high=1, size=(3, d))
a = np.random.normal(loc=3, scale=1, size=(d))
np.random.seed(42)
X, Y, Z, C = getdata(n, exploredpaths, w, a, noiserange=0)

In [85]:
theta = bandit_SGD(X_train, Z_train, C_train, reg=False, epochs=400)
cost = np.sum([np.dot(Y_test[i], solmap(f(X_test[i], theta))) for i in range(n)]) / n
optcost = np.sum([np.dot(Y_test[i], solmap(Y_test[i])) for i in range(n)]) / n

print((cost - optcost)/optcost)

  0%|          | 0/400 [00:00<?, ?it/s]

0.2605548681918659


In [95]:
out

{0: np.float64(0.3465979867228888),
 0.5: np.float64(0.2874042223739838),
 1: np.float64(0.2834523904500432),
 1.5: np.float64(0.2851372495053794),
 2: np.float64(0.2348532551320332)}